**This project belongs to the [Intermediate Machine Learning](https://www.kaggle.com/learn/intermediate-machine-learning) course from Kaggle.**

I present my own solution for this project

---


In this project, we work with data from the [Housing Prices Competition for Kaggle Learn Users](https://www.kaggle.com/c/home-data-for-ml-course). 

(https://storage.googleapis.com/kaggle-media/learn/images/lTJVG4e.png)

we have a test.csv and training.csv set of data. the training.csv contains all the information regarding housing prices. all the information regarding the tables could be found on the website mentioned above. 
Task: Implement a machin learning algorithm to predict the sale price of housing using both dropping and imputing the columns with null values. Use only numeric columns in the algorithm.

In [108]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.impute import SimpleImputer

In [61]:
# reading the data
# full_data is the data we are using for generating our model

full_data = pd.read_csv("C:/Users\kokar\PycharmProjects\ML\house price ML\home-data-for-ml-course/train.csv.gz", index_col='Id')

test_data = pd.read_csv('C:/Users\kokar\PycharmProjects\ML\house price ML\home-data-for-ml-course/test.csv', index_col='Id')

full_data.info() # we can see the data types of the data frames and the null value conditions.

<class 'pandas.core.frame.DataFrame'>
Index: 1460 entries, 1 to 1460
Data columns (total 80 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   MSSubClass     1460 non-null   int64  
 1   MSZoning       1460 non-null   object 
 2   LotFrontage    1201 non-null   float64
 3   LotArea        1460 non-null   int64  
 4   Street         1460 non-null   object 
 5   Alley          91 non-null     object 
 6   LotShape       1460 non-null   object 
 7   LandContour    1460 non-null   object 
 8   Utilities      1460 non-null   object 
 9   LotConfig      1460 non-null   object 
 10  LandSlope      1460 non-null   object 
 11  Neighborhood   1460 non-null   object 
 12  Condition1     1460 non-null   object 
 13  Condition2     1460 non-null   object 
 14  BldgType       1460 non-null   object 
 15  HouseStyle     1460 non-null   object 
 16  OverallQual    1460 non-null   int64  
 17  OverallCond    1460 non-null   int64  
 18  YearBuilt    

In [ ]:
# from the table above we see that the SalePrice does not have any null entry. I separate the feature and target columns, and exclude non-numeric columns

In [73]:

y = full_data.SalePrice  # target
X = full_data.drop( ['SalePrice'], axis=1) # features

# removing non-numeric columns
X = X.select_dtypes(exclude=['object'])

# removing non-numeric columns from test data which includes only features (not the target column)
X_test = test_data.select_dtypes(exclude=['object'])


# Breaking off validation set from training data for fitting the model
X_train, X_valid, y_train, y_valid = train_test_split(X, y, train_size=0.8, test_size=0.2,
                                                      random_state=0)

X_train.shape

(1168, 36)

In [88]:
X_train.head()

,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,BsmtFinSF2,...,GarageArea,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold
Id,,,,,,,,,,,,,,,,,,,,,
619,20,90.0,11694,9,5,2007,2007,452.0,48,0,...,774,0,108,0,0,260,0,0,7,2007
871,20,60.0,6600,5,5,1962,1962,0.0,0,0,...,308,0,0,0,0,0,0,0,8,2009
93,30,80.0,13360,5,7,1921,2006,0.0,713,0,...,432,0,0,44,0,0,0,0,8,2009
818,20,NaN,13265,8,5,2002,2002,148.0,1218,0,...,857,150,59,0,0,0,0,0,7,2008
303,20,118.0,13704,7,5,2001,2002,150.0,0,0,...,843,468,81,0,0,0,0,0,1,2006


In [75]:
# we can see the number of null values for the columns with missing entries
X_train.isnull().sum()[X_train.isnull().sum() != 0]

LotFrontage    212
MasVnrArea       6
GarageYrBlt     58
dtype: int64

In [112]:
# getting the column names with non-null values 

kept_headers = X_train.isnull().any()[X_train.isnull().any() == False].index.to_list()

reduced_X_train = X_train[kept_headers]
reduced_X_valid = X_valid[kept_headers]
y_train.shape

(1168,)

In [113]:
model_removal = RandomForestRegressor(n_estimators=100, random_state=0)
model_removal.fit(reduced_X_train, y_train)
predict_val = model_removal.predict(reduced_X_valid)
MAE = mean_absolute_error(y_valid, predict_val)
print(f'MAE: {MAE}')

MAE: 17837.82570776256


In [115]:
# finding the target values for X_test data
prediction = model_removal.predict(X_test[kept_headers])
output = pd.DataFrame({'Id': X_test.index, 'SalePrice': prediction})
output.head()

,Id,SalePrice
0,1461,126915.50
1,1462,157737.00
2,1463,181556.40
3,1464,184109.00
4,1465,197827.95


In [116]:
# Now, I use the SimpleImputer() module from scikit-learn to impute the missing values

imputer = SimpleImputer()
imputed_X_train = pd.DataFrame(imputer.fit_transform(X_train))
imputed_X_valid = pd.DataFrame(imputer.transform(X_valid))

# Imputation removed column names. so I put them back
imputed_X_train.columns = X_train.columns
imputed_X_valid.columns = X_valid.columns


In [110]:
model_imputation = RandomForestRegressor(n_estimators=100, random_state=0)
model_imputation.fit(imputed_X_train, y_train)
predict_val = model_imputation.predict(imputed_X_valid)
MAE = mean_absolute_error(y_valid, predict_val)
print(f'MAE: {MAE}')

MAE: 18062.894611872147
